<a href="https://colab.research.google.com/github/B3nj4120/B3nj4120/blob/main/Evaluacion1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

In [11]:
df = pd.read_csv('/content/IMDB Dataset.csv')

In [10]:
le = LabelEncoder()
df['sentiment'] = le.fit_transform(df['sentiment'])

In [13]:
X = df['review']
y = df['sentiment']

In [14]:
# Dividimos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Vectorización de texto usando TF-IDF (Limitamos a 5000 palabras para no sobrecargar el MLP)
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train).toarray()
X_test_vec = vectorizer.transform(X_test).toarray()

In [18]:
# Función para crear el modelo, permitiendo variar hiperparámetros
def crear_modelo_mlp(funcion_activacion='relu', tasa_aprendizaje=0.001):
    modelo = Sequential()

    # Capa de entrada y primera capa oculta con regularización L2
    modelo.add(Dense(128, input_shape=(5000,), activation=funcion_activacion, kernel_regularizer=l2(0.001)))
    # Técnica de optimización: Dropout
    modelo.add(Dropout(0.5))

    # Segunda capa oculta
    modelo.add(Dense(64, activation=funcion_activacion, kernel_regularizer=l2(0.001)))
    modelo.add(Dropout(0.5))

    # Capa de salida binaria (función de activación sigmoide para clasificación 0 o 1)
    modelo.add(Dense(1, activation='sigmoid'))

    # Optimizador Adam con tasa de aprendizaje ajustable
    optimizador = tf.keras.optimizers.Adam(learning_rate=tasa_aprendizaje)

    # Función de error 'binary_crossentropy'
    modelo.compile(optimizer=optimizador,
                  loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])
    return modelo

modelo_optimo = crear_modelo_mlp(funcion_activacion='relu', tasa_aprendizaje=0.001)
modelo_optimo.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 128)            │       640,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 648,449 (2.47 MB)

 Trainable params: 648,449 (2.47 MB)

 Non-trainable params: 0 (0.00 B)